# Pipeline C — Import dependency of business investment in Kosovo

**Question:** How much do business investments in Kosovo depend on imports, and which sectors invest most in the domestic economy?

When a business invests (machinery, equipment, construction), the money either leaves the country (imports) or stays with domestic suppliers. The ATK VAT-declaration report already splits investment by origin, so the question comes from the structure of the data itself.

**Run first** (from the project root):
```
python -m phase_1.run_common_preprocessing
python -m phase_1.pipeline_c.run_pipeline_c
```
This notebook only reads and explains the saved outputs.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/schema.py").exists()), None)
if root is None:
    raise FileNotFoundError("Open the notebook inside the project")
sys.path.insert(0, str(root))
from phase_1.pipeline_c import TABLES, FIGURES, VALIDATION, MEASURE_LABELS, ORIGIN_OF, read_table

status = json.loads(VALIDATION.read_text(encoding="utf-8"))
print("Pipeline C status:", status["status"])
pd.set_option("display.float_format", "{:.3f}".format)

## 1. Feature creation: three origin groups, not two

The six monetary columns are grouped by **origin**. The original Excel headers decide the grouping:

| Source header (ATK) | Group |
|---|---|
| Importet investive me normë 18% / 8% | **import** |
| Blerjet investive vendore me normë 18% / 8% | **domestic** |
| *Blerjet dhe importet* investive pa TVSH | **mixed** |
| *Blerjet dhe importet* investive me TVSH jo të zbritshme | **mixed** |

The two "mixed" columns contain *purchases and imports* together, so their origin cannot be separated. Counting them as imports would overstate import dependency. This matters most in VAT-exempt sectors (education, health, finance), where most purchases fall in the non-deductible VAT column.

Three shares are therefore created:
- `import_share = import / (import + domestic)`: the **main measure**, using only amounts whose origin is known
- `import_share_lower = import / total`: if every mixed euro were domestic
- `import_share_upper = (import + mixed) / total`: if every mixed euro were imported

`import_share` always lies between the two bounds.

In [ ]:
display(pd.DataFrame({"measure": list(MEASURE_LABELS), "label": list(MEASURE_LABELS.values()),
                      "origin": [ORIGIN_OF[m] for m in MEASURE_LABELS]}))

## 2. Aggregation: why shares are not computed per row

Most source rows have no investment at all, or only one origin. A per-row share would be undefined (0/0) or trivially 0% or 100%. Shares are therefore computed on **aggregates** (sector × year, municipality × year, national × year), as a ratio of sums, which weights every euro equally.

In [ ]:
display(read_table("row_level_definedness"))
national = read_table("national_classes")
display(national.query("window == 'reported'")[["year", "months", "import_share", "import_share_lower", "import_share_upper", "mixed_share", "import_dominant"]])

## 3. Feature subset selection

Two criteria, both computed on the source rows:
1. **Variance threshold** on the binary indicator "amount is nonzero": `p(1-p) ≥ 0.95·0.05`. A column that is zero in more than 95% of rows is near-constant.
2. **Value share**: the column must hold at least 1% of all invested value.

The 8% import and 8% domestic columns fail, so they are removed from the multivariate (PCA) feature set. They still stay inside the origin totals, so all totals reconcile with the source.

In [ ]:
display(read_table("feature_selection")[["measure", "origin", "nonzero_rate", "indicator_variance", "value_share", "selected", "reason"]])

## 4. Transformation and dimensionality reduction (PCA)

Matrix: sector × complete year (2019–2025), selected attributes. The amounts are extremely right-skewed, so `log1p` is applied, then a z-score, so that no attribute dominates through its scale.

PCA is computed with the SVD of the standardized matrix. Sign convention: PC1 points towards more investment, and later components point towards the import side.

In [ ]:
display(read_table("transformation_skewness"))
display(read_table("pca_explained_variance"))
display(read_table("pca_loadings")[["label", "origin", "loading_PC1", "loading_PC2", "loading_PC3"]])
display(read_table("pca_origin_contrast"))

In [ ]:
display(Image(filename=str(FIGURES / "c5_pca_biplot.png")))

**Reading:** PC1 is a *size* component, since all loadings are positive (bigger sectors invest more in every category). PC2 does **not** separate imports from domestic purchases. It separates the two *mixed* (no VAT / non-deductible VAT) columns from the pure VAT columns, i.e. the **VAT regime** of the sector. Import vs domestic only appears in PC3 (~5%). So the import/domestic split is a real but minor axis of variation; sector size and VAT treatment dominate.

## 5. Binarization and discretization

- `import_dominant = 1` if `import_share > 0.5`
- `origin_class`: mostly domestic (< 33%), mixed (33–67%), mostly import (> 67%)
- `class_robust`: the class is the same whether the mixed amounts are counted as domestic (lower bound) or as imported (upper bound)
- `relative_class`: tertiles of the rank within each year, i.e. *which sectors rely on imports more than the others*. This stays informative even when almost every sector falls in the same absolute class.

In [ ]:
sectors = read_table("sector_pooled_classes").query("window == 'reported' and sector_eligible")
display(sectors.sort_values("import_share")[["primary_sector", "import_share", "import_share_lower", "import_share_upper",
                                             "identified_coverage", "import_dominant", "origin_class", "relative_class", "class_robust"]])
display(read_table("class_counts_by_year"))

**Reading:** no sector is import-dominant (`import_dominant = 0` everywhere) and only Transport reaches the middle class. But only **4 of 19 sectors are `class_robust`** (public administration, real estate, accommodation, administrative services): for the other 15, the class would change if the mixed columns turned out to be mostly imported. The absolute classes therefore summarise the *known-origin* money; the relative tertiles are what separates the sectors.

`identified_coverage` = (import + domestic) / total, i.e. how much of the sector's money the import share actually describes. Education, health and finance are at **16%**: their import share rests on a small part of their investment (see c2 below).

## 6. Sampling: bootstrap confidence intervals

For each sector, the source rows are resampled with replacement **within each year** (stratified bootstrap, 1,000 replicates, seed 42), and the import share is recomputed. The 2.5–97.5 percentiles give a 95% interval.

`top5_records_import_share` shows how much of a sector's imports come from its 5 largest rows. A high value means the estimate depends on a few large investments.

In [ ]:
boot = read_table("bootstrap_ci").query("sector_eligible and status == 'valid'")
display(boot.sort_values("import_share")[["primary_sector", "records", "import_share", "ci_low", "ci_high", "ci_width",
                                          "top5_records_import_share", "stability"]])

**Reading:** 17 of the 19 sectors keep their class in at least 95% of the resamples (`class_stable`), and the other 2 at least keep the same side of 50%. Narrow intervals come with many large records (trade, construction, accommodation); wide ones with few (education, energy, water). Note that the bootstrap measures **random variation only**: it cannot see the unknown origin of the mixed columns, which is a separate and larger uncertainty.

### 6b. Sampling as data reduction: how much data does the import share need?

Bootstrap (above) measures how **stable** a share is. Here the question is different: if we kept only **10% of the rows**, would we still get the same import shares? Three designs, all with the same sample size (~12,400 rows), 50 draws each (seed 42):

1. **Simple random**: every row has the same chance.
2. **Stratified by year × sector**: 10% inside every year-sector group, each sampled row weighted by group size / sampled rows.
3. **Take-all 1% largest + stratified rest**: the 1% largest rows are always kept (weight 1), the rest is stratified as in 2.

**Why design 3:** investment is extremely concentrated. The 1% largest rows hold about half of the identified investment, so the error mainly depends on whether those few rows are drawn. Balancing sectors (design 2) does not control that; a take-all stratum for the largest units (standard in business surveys) does.

In [ ]:
display(read_table("sample_sensitivity_summary")[["method", "level", "sampled_records_mean", "mean_abs_error_pp",
                                                  "p95_abs_error_pp", "max_abs_error_pp", "class_agreement"]])
display(Image(filename=str(FIGURES / "c7_sample_sensitivity.png")))

**Reading:** stratifying by sector alone does not beat a simple random sample. Taking all of the largest rows halves the national error (≈2.2 → 1.0 percentage points) with the same number of rows, and cuts it most in the big sectors (trade, construction, manufacturing, energy). Small sectors such as education stay noisy under every design, so a 10% sample is **not** enough for them: the analysis keeps using the full data, and this stage only documents why.

## 7. Results

### c1 — Import share by sector

In [ ]:
display(Image(filename=str(FIGURES / "c1_sector_import_share.png")))

**Reading:**
- **No sector is import-dominant.** Transport (43%) is the only one in the middle class; it buys vehicles, which Kosovo does not produce.
- Higher import reliance: transport, arts and recreation (33%), information and communication (28%), manufacturing (25%), mining (24%): sectors that invest in **machinery and equipment**.
- Lower: real estate (2%), accommodation, energy and public administration (~10%): sectors that invest in **buildings and local services**.
- **Hatched bars are not comparable:** education, health and finance know the origin of only 16% of their money. Education's 32% must not be read as "education relies on imports".

### c2 — Import, domestic or not separable

In [ ]:
display(Image(filename=str(FIGURES / "c2_sector_origin_composition.png")))

**Reading:** this chart is the caveat for c1. The grey part (mixed columns) is small for real estate, accommodation and administrative services, but covers **80–85%** of education, health and finance. These activities are generally VAT-exempt, so the VAT they pay on purchases cannot be deducted and lands in the *non-deductible VAT* column, whose origin is unknown. This is an interpretation of the VAT rules, not something the data states, but PCA (section 4) finds the same group independently: these three sectors are the cluster at the bottom of PC2.

### c3 — National import share over time

In [ ]:
display(Image(filename=str(FIGURES / "c3_national_import_share.png")))

**Reading:**
- About **1 euro in 5** of known-origin investment is imported: 25% in 2019, **16–17% in 2020–2022**, back to 22% in 2024 and ~20% in 2025–2026.
- The 2020–2022 dip coincides with COVID-19 and the supply-chain disruptions; the data show the timing, not the cause.
- The full-year (blue) and January–June (orange) lines almost coincide, so seasonality does not distort the comparison and the half-year 2026 value can be read against the others.
- The grey band is wide (~10% to ~60%) and contains the 50% line: if the mixed money were mostly imported, imports could approach half of all investment. The safe statement is a **range**, with ~20% as the value for known-origin money.

### c4 — Municipal ranking

In [ ]:
display(Image(filename=str(FIGURES / "c4_municipal_ranking.png")))
display(read_table("municipal_ranking")[["side", "rank", "municipality", "import_share", "identified_total",
                                         "identified_coverage", "source_records", "origin_class"]])

**Reading:**
- Most domestic: **Graçanicë 12%, Prishtinë 15%**, then Zveçan, Novobërdë and Gjakovë (17%). Prishtina and its surroundings have the largest market of domestic suppliers.
- Most import-oriented: **Kllokot 44%, Dragash and Partesh 40%**, Deçan 38%, Mamushë 37%. No municipality passes 50%.
- The import side is dominated by **small municipalities**: Partesh has 1.05 M EUR of known-origin investment against Prishtina's 2.2 billion. In a small base a few large purchases move the share a lot, so these rankings are less stable than the national or sector figures (check `identified_total` in the table).

### c6 — Municipal map

Left: the import share of every municipality, coloured around the Kosovo average (blue = less import-dependent than the average, red = more). Right: the same shares as equal-frequency tertiles (the relative classes of stage 04). Municipalities with less than 1 M EUR of identified-origin investment are grey, as in the ranking. The boundaries, hash check and name crosswalk are the same verified reference that Pipeline B uses.

In [ ]:
display(Image(filename=str(FIGURES / "c6_municipal_import_share_map.png")))

**Reading:** the map shows a spatial pattern that the ranking hides. The centre (Prishtinë, Graçanicë, Fushë Kosovë, Lipjan) is below the average. Two above-average areas stand out: the **south-west** (Dragash, Rahovec, Suharekë, Mamushë, Deçan, Junik) and the **east** (Kamenicë, Partesh, Kllokot).

### Summary
1. Business investment in Kosovo is **mostly domestic**: about 1 euro in 5 of known-origin investment is imported (15–25% per year).
2. Sectors that buy **machinery and equipment** import more; sectors that invest in **buildings and local services** import less.
3. **Prishtina and its surroundings buy domestically**; the south-west and the east rely more on imports.
4. The main reservation is the **mixed money** (~40% of the total, over 80% in VAT-exempt sectors): the true import share lies within the lower/upper bounds.
5. The results are statistically stable (17 of 19 sectors keep their class in the bootstrap), and the sampling test shows why the full data are needed.

## 8. Limitations
- About 40% of investment value sits in the two mixed columns, so the true import share is only known within the lower/upper bounds. Only 4 of 19 sectors keep the same class across those bounds (`class_robust`).
- Education, health and finance know the origin of only ~16% of their money (`identified_coverage` < 25%): their import share is computed but flagged as not comparable.
- The bootstrap intervals measure random variation, not the unknown origin of the mixed columns.
- Small municipalities (around 1 M EUR of known-origin investment) can move a lot with a few purchases; the 1 M EUR threshold only removes the smallest three.
- 2026 covers January–June only: compare it through the `jan_jun` window, and leave it out of the PCA.
- Negative adjustments are kept in the totals; aggregates with a negative component get an undefined share instead of a corrected one.
- "Mungon aktiviteti" and "Person Fizik" are not real sectors, so they are excluded from sector comparisons but kept in national and municipal totals.
- The monetary unit is assumed to be EUR (Kosovo's currency); the ATK file does not state it.